# DKVMN Next-Item
Notebook Colab para **DKVMN Next-Item**.


In [ ]:
from pathlib import Path
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/GuilhermeDesoler/ai-core.git'
REPO_DIR = Path('/content/ai-core')
if IN_COLAB:
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd', '/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    REPO_DIR = Path.cwd()
SRC_PATH = REPO_DIR / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('Repo:', REPO_DIR)
print('SRC_PATH:', SRC_PATH)


In [ ]:
from pathlib import Path
import shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount opcional:', e)
PROJECT_ROOT = REPO_DIR
DATA_TARGET = PROJECT_ROOT / 'data'
DATA_TARGET.mkdir(parents=True, exist_ok=True)
SOURCE_RAW_DIR = Path('/content/drive/MyDrive/data/raw')
SOURCE_PROCESSED_DIR = Path('/content/drive/MyDrive/data/processed')
# Descomente se quiser copiar do Drive:
# shutil.copytree(SOURCE_RAW_DIR, DATA_TARGET / 'raw', dirs_exist_ok=True)
# shutil.copytree(SOURCE_PROCESSED_DIR, DATA_TARGET / 'processed', dirs_exist_ok=True)
for p in [Path('data/raw/answers.json'), Path('data/processed/sequences/user_sequences.json')]:
    print(p, p.exists())


In [ ]:
# Opcional: regenere os artefatos base e sequências por sessão.
# get_ipython().system('python run_data_pipeline.py')
# get_ipython().system('python scripts/build_user_session_sequences.py')


In [ ]:
import subprocess, re, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
SCRIPT_PATH = 'scripts/train_eval_dkvmn_next_item.py'
SUMMARY_NAME = 'dkvmn_next_item'
cmd = f'PYTHONPATH=./src python scripts/train_eval_dkvmn_next_item.py'
print('Running:', cmd)
result = subprocess.run(cmd, shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Command failed with code {result.returncode}')
pattern = re.compile(r'Epoch\s+(\d+)(?:/(\d+))?\s*\|\s*Train Loss:\s*([0-9.]+)(?:\s*\|\s*Train AUC:\s*([0-9.]+))?\s*\|\s*Val Loss:\s*([0-9.]+)\s*\|\s*Val AUC:\s*([0-9.]+)\s*\|\s*Val Acc:\s*([0-9.]+)\s*\|\s*LR:\s*([0-9.eE+-]+)')
rows=[]
for line in result.stdout.splitlines():
    m = pattern.search(line)
    if m:
        rows.append({'epoch': int(m.group(1)), 'train_loss': float(m.group(3)), 'train_auc': float(m.group(4)) if m.group(4) else None, 'val_loss': float(m.group(5)), 'val_auc': float(m.group(6)), 'val_acc': float(m.group(7)), 'lr': float(m.group(8))})
history_df = pd.DataFrame(rows)
history_df


In [ ]:
fig = plt.figure(figsize=(10,5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
plt.title(SUMMARY_NAME + ' - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
if history_df['train_auc'].notna().any():
    fig = plt.figure(figsize=(10,5))
    plt.plot(history_df['epoch'], history_df['train_auc'], label='Train AUC')
    plt.plot(history_df['epoch'], history_df['val_auc'], label='Val AUC')
    plt.title(SUMMARY_NAME + ' - AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
saved_match = re.search(r'Saved run to\s+(.+)', result.stdout)
run_dir = Path(saved_match.group(1).strip()) if saved_match else None
metrics = {}
if run_dir and (run_dir / 'metrics.json').exists():
    metrics = json.loads((run_dir / 'metrics.json').read_text())
metrics
summary_dir = Path('artifacts/colab_summaries')
summary_dir.mkdir(parents=True, exist_ok=True)
(summary_dir / f'dkvmn_next_item_history.csv').write_text(history_df.to_csv(index=False))
(summary_dir / f'dkvmn_next_item_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Saved summary files to', summary_dir)
